# UniMol Fine-tuning for Perovskite Additive Prediction

This notebook provides a complete workflow for fine-tuning UniMol models on perovskite solar cell additive data with Optuna hyperparameter optimization.

## Contents
1. Environment Setup
2. Data Loading and Preparation
3. Basic Model Training
4. Hyperparameter Optimization with Optuna
5. Model Evaluation
6. Results Visualization

## 1. Environment Setup

In [ ]:
# Install required packages (run once)
# !pip install unimol-tools optuna optuna-dashboard pandas scikit-learn matplotlib seaborn

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Set UniMol weights directory
os.environ['UNIMOL_WEIGHT_DIR'] = os.path.expanduser('~/.local/lib/python3.10/site-packages/unimol_tools/weights')

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from scipy.stats import pearsonr, spearmanr

# UniMol imports
from unimol_tools import MolTrain, MolPredict

# Optuna for hyperparameter optimization
import optuna
from optuna.samplers import TPESampler

print("All packages imported successfully!")
print(f"UniMol weights directory: {os.environ.get('UNIMOL_WEIGHT_DIR')}")

## 2. Data Loading and Preparation

In [ ]:
# Define paths to data splits
DATA_BASE_PATH = '../baselines/data'
SPLIT_SEEDS = [0, 1, 2, 3, 4]

# Load a specific split
def load_data_split(seed=0):
    """Load train and test data for a specific split seed."""
    split_path = os.path.join(DATA_BASE_PATH, f'split_seed_{seed}')
    
    train_df = pd.read_csv(os.path.join(split_path, 'train.csv'))
    test_df = pd.read_csv(os.path.join(split_path, 'test.csv'))
    
    print(f"Split seed {seed}:")
    print(f"  Training samples: {len(train_df)}")
    print(f"  Test samples: {len(test_df)}")
    
    return train_df, test_df

# Load split seed 0
train_df, test_df = load_data_split(seed=0)
train_df.head()

In [ ]:
# Prepare data for UniMol (needs SMILES and TARGET columns)
def prepare_unimol_data(df, smiles_col='SMILES', target_col='TARGET'):
    """Prepare DataFrame for UniMol training."""
    required_cols = [smiles_col, target_col]
    missing_cols = [col for col in required_cols if col not in df.columns]
    
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")
    
    # Create clean copy with only required columns
    unimol_df = df[[smiles_col, target_col]].copy()
    unimol_df.columns = ['SMILES', 'TARGET']
    
    return unimol_df

# Prepare training data
train_unimol = prepare_unimol_data(train_df)
test_unimol = prepare_unimol_data(test_df)

print("Training data sample:")
print(train_unimol.head())
print(f"\nTarget statistics:")
print(train_unimol['TARGET'].describe())

In [ ]:
# Visualize target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(train_unimol['TARGET'], bins=30, alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Delta PCE (TARGET)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Training Set Target Distribution')

# Box plot
axes[1].boxplot([train_unimol['TARGET'], test_unimol['TARGET']], 
                labels=['Train', 'Test'])
axes[1].set_ylabel('Delta PCE (TARGET)')
axes[1].set_title('Target Distribution Comparison')

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150)
plt.show()

## 3. Basic Model Training

In [ ]:
# Define default training parameters
default_params = {
    'task': 'regression',
    'data_type': 'molecule',
    'model_name': 'unimolv2',
    'model_size': '84m',
    'epochs': 100,
    'batch_size': 16,
    'learning_rate': 1e-4,
    'early_stopping': 20,
    'metrics': 'r2',
    'split': 'random',
    'kfold': 5,
    'remove_hs': False,
    'random_state': 42,
    'target_normalize': 'auto',
    'max_norm': 5.0,
    'warmup_ratio': 0.03,
}

print("Default Training Parameters:")
for key, value in default_params.items():
    print(f"  {key}: {value}")

In [ ]:
# Train a basic model
import tempfile

# Create save directory
save_dir = './unimol_baseline'
os.makedirs(save_dir, exist_ok=True)

# Save training data to temp file (UniMol expects CSV input)
train_csv_path = os.path.join(save_dir, 'train_data.csv')
train_unimol.to_csv(train_csv_path, index=False)

# Initialize trainer
clf = MolTrain(
    save_path=save_dir,
    **default_params
)

# Train the model
print("Starting training...")
clf.fit(train_csv_path)
print("Training completed!")

In [ ]:
# Make predictions
# Save test data
test_csv_path = os.path.join(save_dir, 'test_data.csv')
test_unimol.to_csv(test_csv_path, index=False)

# Load predictor
predictor = MolPredict(load_model=save_dir)

# Predict
train_pred = predictor.predict(train_csv_path)
test_pred = predictor.predict(test_csv_path)

print(f"Train predictions shape: {train_pred.shape}")
print(f"Test predictions shape: {test_pred.shape}")

In [ ]:
# Evaluate predictions
def evaluate_predictions(y_true, y_pred, name='Model'):
    """Calculate and print evaluation metrics."""
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    pearson = pearsonr(y_true, y_pred)[0]
    spearman = spearmanr(y_true, y_pred)[0]
    
    print(f"\n{name} Evaluation:")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE:  {mae:.4f}")
    print(f"  R²:   {r2:.4f}")
    print(f"  Pearson: {pearson:.4f}")
    print(f"  Spearman: {spearman:.4f}")
    
    return {
        'rmse': rmse,
        'mae': mae,
        'r2': r2,
        'pearson': pearson,
        'spearman': spearman
    }

# Evaluate
train_metrics = evaluate_predictions(train_unimol['TARGET'].values, train_pred, 'Training')
test_metrics = evaluate_predictions(test_unimol['TARGET'].values, test_pred, 'Test')

In [ ]:
# Plot predictions vs actual
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Training set
axes[0].scatter(train_unimol['TARGET'], train_pred, alpha=0.5)
axes[0].plot([train_unimol['TARGET'].min(), train_unimol['TARGET'].max()],
             [train_unimol['TARGET'].min(), train_unimol['TARGET'].max()],
             'r--', lw=2)
axes[0].set_xlabel('Actual Delta PCE')
axes[0].set_ylabel('Predicted Delta PCE')
axes[0].set_title(f'Training Set (R²={train_metrics["r2"]:.3f})')

# Test set
axes[1].scatter(test_unimol['TARGET'], test_pred, alpha=0.5)
axes[1].plot([test_unimol['TARGET'].min(), test_unimol['TARGET'].max()],
             [test_unimol['TARGET'].min(), test_unimol['TARGET'].max()],
             'r--', lw=2)
axes[1].set_xlabel('Actual Delta PCE')
axes[1].set_ylabel('Predicted Delta PCE')
axes[1].set_title(f'Test Set (R²={test_metrics["r2"]:.3f})')

plt.tight_layout()
plt.savefig('prediction_vs_actual.png', dpi=150)
plt.show()

## 4. Hyperparameter Optimization with Optuna

In [ ]:
# Define the objective function for Optuna
def objective(trial, train_df, save_base_dir):
    """
    Optuna objective function for hyperparameter optimization.
    
    Parameters:
        trial: Optuna trial object
        train_df: Training DataFrame
        save_base_dir: Base directory for saving models
    
    Returns:
        float: Cross-validation R² score (to maximize)
    """
    # Define search space
    params = {
        'task': 'regression',
        'data_type': 'molecule',
        'model_name': trial.suggest_categorical('model_name', ['unimolv1', 'unimolv2']),
        'model_size': trial.suggest_categorical('model_size', ['84m', '164m', '310m']),
        'batch_size': trial.suggest_categorical('batch_size', [4, 8, 16, 32]),
        'learning_rate': trial.suggest_float('learning_rate', 1e-6, 1e-3, log=True),
        'epochs': trial.suggest_int('epochs', 20, 150),
        'early_stopping': trial.suggest_int('early_stopping', 5, 30),
        'kfold': trial.suggest_categorical('kfold', [3, 5]),
        'max_norm': trial.suggest_float('max_norm', 1.0, 10.0),
        'warmup_ratio': trial.suggest_float('warmup_ratio', 0.0, 0.1),
        'remove_hs': trial.suggest_categorical('remove_hs', [True, False]),
        'split': trial.suggest_categorical('split', ['random']),
        'metrics': 'r2',
        'random_state': 42,
        'target_normalize': trial.suggest_categorical('target_normalize', ['auto', 'standard', 'minmax']),
    }
    
    # Create unique save directory for this trial
    trial_save_dir = os.path.join(save_base_dir, f'trial_{trial.number}')
    os.makedirs(trial_save_dir, exist_ok=True)
    
    # Save training data
    train_csv_path = os.path.join(trial_save_dir, 'train_data.csv')
    train_df.to_csv(train_csv_path, index=False)
    
    try:
        # Initialize and train
        clf = MolTrain(
            save_path=trial_save_dir,
            **params
        )
        clf.fit(train_csv_path)
        
        # Read cross-validation results
        cv_results_path = os.path.join(trial_save_dir, 'metric.result')
        if os.path.exists(cv_results_path):
            with open(cv_results_path, 'r') as f:
                content = f.read()
                # Parse R² score from results
                # This depends on the actual output format
                
        # Use predictor to get CV score
        predictor = MolPredict(load_model=trial_save_dir)
        predictions = predictor.predict(train_csv_path)
        
        # Calculate R² on training data as proxy
        r2 = r2_score(train_df['TARGET'].values, predictions)
        
        return r2
        
    except Exception as e:
        print(f"Trial {trial.number} failed: {e}")
        return -1.0  # Return poor score for failed trials

print("Optuna objective function defined.")

In [ ]:
# Run Optuna optimization
def run_optuna_optimization(train_df, n_trials=50, n_jobs=1):
    """
    Run Optuna hyperparameter optimization.
    
    Parameters:
        train_df: Training DataFrame
        n_trials: Number of optimization trials
        n_jobs: Number of parallel jobs (1 for GPU constraint)
    
    Returns:
        optuna.Study: Completed study object
    """
    # Create study
    study = optuna.create_study(
        study_name='unimol_perovskite_optimization',
        storage='sqlite:///optuna_study.db',
        direction='maximize',  # Maximize R²
        sampler=TPESampler(seed=42),
        load_if_exists=True
    )
    
    # Create base directory for trials
    optuna_save_dir = './optuna_trials'
    os.makedirs(optuna_save_dir, exist_ok=True)
    
    # Run optimization
    study.optimize(
        lambda trial: objective(trial, train_df, optuna_save_dir),
        n_trials=n_trials,
        n_jobs=n_jobs,
        show_progress_bar=True
    )
    
    return study

# Run optimization (adjust n_trials as needed)
N_TRIALS = 20  # Start with fewer trials for testing

print(f"Starting Optuna optimization with {N_TRIALS} trials...")
study = run_optuna_optimization(train_unimol, n_trials=N_TRIALS)

In [ ]:
# Display optimization results
print("\n" + "="*50)
print("OPTIMIZATION RESULTS")
print("="*50)

print(f"\nBest trial: {study.best_trial.number}")
print(f"Best R² score: {study.best_value:.4f}")
print("\nBest hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

In [ ]:
# Plot optimization history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Optimization history
trials = study.trials
trial_numbers = [t.number for t in trials]
trial_values = [t.value for t in trials if t.value is not None]
trial_numbers_valid = [t.number for t in trials if t.value is not None]

axes[0].scatter(trial_numbers_valid, trial_values, alpha=0.6)
axes[0].axhline(y=study.best_value, color='r', linestyle='--', label=f'Best: {study.best_value:.3f}')
axes[0].set_xlabel('Trial Number')
axes[0].set_ylabel('R² Score')
axes[0].set_title('Optimization History')
axes[0].legend()

# Parameter importance (if enough trials)
if len(study.trials) >= 10:
    try:
        importance = optuna.importance.get_param_importances(study)
        params = list(importance.keys())
        values = list(importance.values())
        
        axes[1].barh(params, values)
        axes[1].set_xlabel('Importance')
        axes[1].set_title('Hyperparameter Importance')
    except:
        axes[1].text(0.5, 0.5, 'Not enough trials\nfor importance analysis',
                    ha='center', va='center', transform=axes[1].transAxes)
else:
    axes[1].text(0.5, 0.5, 'Not enough trials\nfor importance analysis',
                ha='center', va='center', transform=axes[1].transAxes)

plt.tight_layout()
plt.savefig('optuna_results.png', dpi=150)
plt.show()

## 5. Train Final Model with Best Parameters

In [ ]:
# Train final model with best parameters
best_params = study.best_params.copy()
best_params.update({
    'task': 'regression',
    'data_type': 'molecule',
    'metrics': 'r2',
    'random_state': 42,
})

print("Training final model with best parameters:")
for key, value in best_params.items():
    print(f"  {key}: {value}")

In [ ]:
# Create final model directory
final_save_dir = './unimol_best_model'
os.makedirs(final_save_dir, exist_ok=True)

# Save training and test data
final_train_csv = os.path.join(final_save_dir, 'train.csv')
final_test_csv = os.path.join(final_save_dir, 'test.csv')
train_unimol.to_csv(final_train_csv, index=False)
test_unimol.to_csv(final_test_csv, index=False)

# Train final model
final_clf = MolTrain(
    save_path=final_save_dir,
    **best_params
)

print("\nTraining final model...")
final_clf.fit(final_train_csv)
print("Training completed!")

In [ ]:
# Evaluate final model
final_predictor = MolPredict(load_model=final_save_dir)

final_train_pred = final_predictor.predict(final_train_csv)
final_test_pred = final_predictor.predict(final_test_csv)

final_train_metrics = evaluate_predictions(
    train_unimol['TARGET'].values, final_train_pred, 'Final Training'
)
final_test_metrics = evaluate_predictions(
    test_unimol['TARGET'].values, final_test_pred, 'Final Test'
)

## 6. Results Summary and Comparison

In [ ]:
# Compare baseline vs optimized model
comparison_df = pd.DataFrame({
    'Metric': ['RMSE', 'MAE', 'R²', 'Pearson', 'Spearman'],
    'Baseline (Train)': [train_metrics['rmse'], train_metrics['mae'], 
                         train_metrics['r2'], train_metrics['pearson'], train_metrics['spearman']],
    'Baseline (Test)': [test_metrics['rmse'], test_metrics['mae'],
                        test_metrics['r2'], test_metrics['pearson'], test_metrics['spearman']],
    'Optimized (Train)': [final_train_metrics['rmse'], final_train_metrics['mae'],
                          final_train_metrics['r2'], final_train_metrics['pearson'], final_train_metrics['spearman']],
    'Optimized (Test)': [final_test_metrics['rmse'], final_test_metrics['mae'],
                         final_test_metrics['r2'], final_test_metrics['pearson'], final_test_metrics['spearman']],
})

print("\n" + "="*70)
print("MODEL COMPARISON")
print("="*70)
print(comparison_df.to_string(index=False))

In [ ]:
# Save results
results_summary = {
    'best_params': best_params,
    'baseline_test_r2': test_metrics['r2'],
    'optimized_test_r2': final_test_metrics['r2'],
    'improvement': final_test_metrics['r2'] - test_metrics['r2']
}

import json
with open('results_summary.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print("\nResults saved to results_summary.json")
print(f"\nR² improvement: {results_summary['improvement']:.4f}")

In [ ]:
# Final visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Baseline predictions
axes[0, 0].scatter(train_unimol['TARGET'], train_pred, alpha=0.5)
axes[0, 0].plot([train_unimol['TARGET'].min(), train_unimol['TARGET'].max()],
                [train_unimol['TARGET'].min(), train_unimol['TARGET'].max()], 'r--')
axes[0, 0].set_title(f'Baseline - Training (R²={test_metrics["r2"]:.3f})')
axes[0, 0].set_xlabel('Actual')
axes[0, 0].set_ylabel('Predicted')

axes[0, 1].scatter(test_unimol['TARGET'], test_pred, alpha=0.5)
axes[0, 1].plot([test_unimol['TARGET'].min(), test_unimol['TARGET'].max()],
                [test_unimol['TARGET'].min(), test_unimol['TARGET'].max()], 'r--')
axes[0, 1].set_title(f'Baseline - Test (R²={test_metrics["r2"]:.3f})')
axes[0, 1].set_xlabel('Actual')
axes[0, 1].set_ylabel('Predicted')

# Optimized predictions
axes[1, 0].scatter(train_unimol['TARGET'], final_train_pred, alpha=0.5, c='green')
axes[1, 0].plot([train_unimol['TARGET'].min(), train_unimol['TARGET'].max()],
                [train_unimol['TARGET'].min(), train_unimol['TARGET'].max()], 'r--')
axes[1, 0].set_title(f'Optimized - Training (R²={final_train_metrics["r2"]:.3f})')
axes[1, 0].set_xlabel('Actual')
axes[1, 0].set_ylabel('Predicted')

axes[1, 1].scatter(test_unimol['TARGET'], final_test_pred, alpha=0.5, c='green')
axes[1, 1].plot([test_unimol['TARGET'].min(), test_unimol['TARGET'].max()],
                [test_unimol['TARGET'].min(), test_unimol['TARGET'].max()], 'r--')
axes[1, 1].set_title(f'Optimized - Test (R²={final_test_metrics["r2"]:.3f})')
axes[1, 1].set_xlabel('Actual')
axes[1, 1].set_ylabel('Predicted')

plt.tight_layout()
plt.savefig('final_comparison.png', dpi=150)
plt.show()

print("\nNotebook completed successfully!")
print("Generated files:")
print("  - target_distribution.png")
print("  - prediction_vs_actual.png")
print("  - optuna_results.png")
print("  - final_comparison.png")
print("  - results_summary.json")